# 04 Structured Output Portability (LiteLLM, 2026)

## What This Lesson Is
Normalize outputs into a strict schema so downstream logic is model/provider agnostic.

## Scientific Lens
- Concept: Schema-first interoperability across model providers
- Measure: Valid schema parse rate
- Validity Limit: Schema-valid output may still contain factual errors.


## How It Works
1. Define strict schema contract.
2. Validate deterministic payloads.
3. Parse live model output through the same validator.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Schema keys:", ["action", "owner", "priority"])


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import json

REQUIRED = {"action", "owner", "priority"}
ALLOWED_PRIORITY = {"low", "medium", "high"}

def validate(payload: str):
    data = json.loads(payload)
    missing = REQUIRED - set(data)
    if missing:
        raise ValueError(f"missing fields: {missing}")
    if data["priority"] not in ALLOWED_PRIORITY:
        raise ValueError("invalid priority")
    return data

sample = '{"action":"rotate_keys","owner":"platform","priority":"high"}'
parsed = validate(sample)
print(parsed)
assert parsed["owner"] == "platform"


In [ ]:
# Live Demo
import json
import os

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live schema demo: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live schema demo: OPENAI_API_KEY not set.")
    else:
        prompt = "Return strict JSON with keys action, owner, priority. No prose."
        r = completion(
            model="openai/gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            api_key=api_key,
            timeout=20,
        )
        text = r.choices[0].message.content.strip()
        print(text)
        try:
            obj = json.loads(text)
            print("parsed:", obj)
        except Exception as exc:
            print(f"Parse failed ({exc}); apply repair strategy before downstream use.")


## Applied Labs
1. Add a `due_date` optional field and update schema validation accordingly.
2. Test parser behavior on malformed JSON and implement a repair fallback.
3. Record schema pass/fail counts over 20 live generations.

## Validation Checklist
- Schema validation rejects missing required fields.
- Priority field is constrained to explicit enum values.
- Live generation path has safe parse-failure handling.

## Further Reading
- [LiteLLM Structured Output](https://docs.litellm.ai/docs/completion)
- [JSON Schema Core](https://json-schema.org/specification)
- [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)
